In [1]:
# Test script 1

In [3]:
# Plot time series of OSDMA8, globally and regionally

In [1]:
import xarray as xr
import regionmask
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.gridspec import GridSpec

In [7]:
# === Removing ocean ===
def land_filter(da):
    # Load in land fraction dataset
    land_110 = regionmask.defined_regions.natural_earth_v4_1_0.land_110
    da = da.where(land_110.mask_3D(da).squeeze())
    return da


# === Figure sizing ===
def autosize_figure(nrows, ncolumns, scale_factor=1, xscale_factor=1, yscale_factor=1):
    xwidth = (ncolumns+0.67) * 5.0 * scale_factor * xscale_factor
    ylength = (nrows+0.67) * 3.6 * scale_factor * yscale_factor
    return (xwidth, ylength)


# === Concatenate ensemble members ===
def merge_global_mean_ensembles(VAR_DIR, VAR, land_mask=False):
    """ land_filter=True allows for the land filter function to remove the ocean """
    ARISE_ens = []
    SSP245_ens = []

    for ens_num in range(1, 11):
        arise = xr.open_dataarray(f"{VAR_DIR}{VAR}_CESM2_ARISE_{ens_num:02d}_2035-2069.nc")
        ssp245 = xr.open_dataarray(f"{VAR_DIR}{VAR}_CESM2_SSP245_{ens_num:02d}_2020-2069.nc")

        if land_mask is True:
            print("Filtering ocean")
            arise = land_filter(arise)
            ssp245 = land_filter(ssp245)

        ARISE_ens.append(arise.mean(dim=("lat", "lon")))
        SSP245_ens.append(ssp245.mean(dim=("lat", "lon")))

    arise_da = xr.concat(ARISE_ens, dim=xr.DataArray(np.arange(1, len(ARISE_ens)+1), dims="ensemble", name="ensemble"))
    ssp245_da = xr.concat(SSP245_ens, dim=xr.DataArray(np.arange(1, len(SSP245_ens)+1), dims="ensemble", name="ensemble"))

    return arise_da, ssp245_da

In [ ]:
# === Path config ===
OSDMA8_DIR = "/glade/work/awells/air_quality/CESM/ozone/OSDMA8_BC/"
SCENARIOS = ["ARISE", "SSP245"]


# === Main loop ===
arise_land, ssp245_land = merge_global_mean_ensembles(OSDMA8_DIR, "OSDMA8_BC_popgrid", land_mask=True)

Filtering ocean


In [ ]:
arise_land

In [ ]:
fig = plt.figure(figsize=autosize_figure(2, 1, xscale_factor=1.5))
gs = GridSpec(2, 1) # rows, columns

ax1 = fig.add_subplot(gs[0, 0])
for ens_num in range(1, 11):
    ssp245_land.sel(ensemble=ens_num, quantile="median").plot(color='grey', alpha=0.4, linewidth=0.5)
    arise_land.sel(ensemble=ens_num, quantile="median").plot(color='tab:purple', alpha=0.4, linewidth=0.5)
ssp245_land.mean("ensemble").sel(quantile="median").plot(color='grey', label="SSP2-4.5")
arise_land.mean("ensemble").sel(quantile="median").plot(color='tab:purple', label="ARISE-SAI-1.5")
plt.title("Global OSDMA8")
plt.legend()

ax2 = fig.add_subplot(gs[1, 0])
ssp245_time = ssp245_land.sel(year=slice(2035, 2069))
for ens_num in range(1, 11):
    (arise_land - ssp245_time).sel(ensemble=ens_num, quantile="median").plot(color='k', alpha=0.4, linewidth=0.5)
(arise_land - ssp245_time).mean("ensemble").sel(quantile="median").plot(color='k')
plt.axhline(0, color='k', linestyle='--')
plt.title("Difference in Global OSDMA8")